# How to Measure School Quality from Exam Results

**Context:** The Warsaw OKE district publishes results of the Polish 8th-grade exam (egzamin ósmoklasisty)
for every school each year. We have data from 2021 to 2025.

**Goal of this notebook:** Decide *which single number* best represents a school's quality,
given the data we have — and show the reasoning behind that choice.

**Why this matters:** The number we choose will colour the dots on a public map.
A bad choice can unfairly label a school as weak (or strong) due to statistical noise,
not real differences in teaching quality.

**Structure:**
1. Load data and show why only 3 subjects are usable
2. Choose the best per-year metric (mean, median, percentile, or difference from national)
3. Choose the best way to combine multiple years
4. Identify schools that are *consistently* in the top
5. Combine all 3 subjects into one score
6. Final metric definition

## 0. Setup

In [ ]:
# Run with: uv run jupyter lab
from pathlib import Path
import re
import unicodedata
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('../data/egzamin-osmoklasisty')

# Subjects with enough data for analysis
CORE_SUBJECTS = ['jezyk polski', 'matematyka', 'jezyk angielski']
SUBJECT_LABELS = {
    'jezyk polski':    'Polish',
    'matematyka':      'Maths',
    'jezyk angielski': 'English',
}
SUBJECT_COLORS = {
    'jezyk polski':    '#2166ac',
    'matematyka':      '#d6604d',
    'jezyk angielski': '#1a9850',
}

# Jackknife shrinkage constant (see Section 3)
SHRINKAGE_K = 15

## 1. Load data

Each year's results come in a separate Excel file. We load them all and stack them
into one DataFrame where each row is one school in one year.

In [ ]:
YEAR_FILE_RE = re.compile(r'^\d{4}', re.IGNORECASE)


def normalize_text(text: str) -> str:
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = text.replace('\n', ' ')
    return re.sub(r'\s+', ' ', text).lower().strip()


def clean_header_value(value: object) -> str:
    if value is None:
        return ''
    text = str(value)
    if text.startswith('Unnamed:') or text == 'nan':
        return ''
    return re.sub(r'\s+', ' ', text.replace('\n', ' ')).strip()


def normalize_multiindex_columns(columns: pd.MultiIndex) -> pd.MultiIndex:
    lvl0_values, lvl1_values = [], []
    last_lvl0 = 'meta'
    for raw0, raw1 in columns.to_list():
        lvl0_raw = clean_header_value(raw0)
        lvl1_raw = clean_header_value(raw1)
        if lvl0_raw:
            last_lvl0 = normalize_text(lvl0_raw)
        lvl0_values.append(last_lvl0)
        lvl1_values.append(normalize_text(lvl1_raw) if lvl1_raw else 'value')
    return pd.MultiIndex.from_arrays([lvl0_values, lvl1_values], names=['subject', 'metric'])


def read_exam_file(path: Path) -> pd.DataFrame:
    year = int(YEAR_FILE_RE.match(path.name).group(0))
    df = pd.read_excel(path, sheet_name='SAS', header=[0, 1])
    df.columns = normalize_multiindex_columns(df.columns)
    df[('meta', 'year')] = year
    return df


files = sorted(p for p in DATA_DIR.glob('*.xlsx*') if YEAR_FILE_RE.match(p.name))
assert files, f'No data files found in {DATA_DIR}'

raw_frames = [read_exam_file(p) for p in files]
df_raw = pd.concat(raw_frames, ignore_index=True)

years = [int(YEAR_FILE_RE.match(p.name).group(0)) for p in files]
print(f'Loaded {len(files)} files: {years}')
print(f'Total rows: {len(df_raw):,}  (one row = one school × one year)')

In [ ]:
# Build a flat working DataFrame with consistent column names
# Each subject contributes 3 columns: n_students, median_pct, mean_pct
ALL_SUBJECTS = [
    'jezyk polski', 'matematyka', 'jezyk angielski',
    'jezyk francuski', 'jezyk hiszpanski', 'jezyk niemiecki',
    'jezyk rosyjski', 'jezyk wloski',
]

def col(subject: str, metric: str) -> tuple:
    return (subject, metric)

records = {}
records['rspo']       = df_raw[('meta', 'rspo')]
records['year']       = df_raw[('meta', 'year')]
records['school_name'] = (
    df_raw[('meta', 'nazwa szkoły')]
    if ('meta', 'nazwa szkoły') in df_raw.columns
    else df_raw[('meta', 'nazwa szkoly')]
    if ('meta', 'nazwa szkoly') in df_raw.columns
    else pd.Series(dtype=str)
)
records['is_public']  = df_raw.get(('meta', 'czy publiczna'), pd.Series(dtype=str))
records['gmina']      = df_raw.get(('meta', 'gmina - nazwa'), pd.Series(dtype=str))
records['powiat']     = df_raw.get(('meta', 'powiat - nazwa'), pd.Series(dtype=str))

# Try both Polish and normalized subject names
subject_col_map = {
    'jezyk polski':     'jezyk polski',
    'matematyka':       'matematyka',
    'jezyk angielski':  'jezyk angielski',
    'jezyk francuski':  'jezyk francuski',
    'jezyk hiszpanski': 'jezyk hiszpanski',
    'jezyk niemiecki':  'jezyk niemiecki',
    'jezyk rosyjski':   'jezyk rosyjski',
    'jezyk wloski':     'jezyk wloski',
}

for key, src in subject_col_map.items():
    short = key.replace('jezyk ', '').replace(' ', '_')
    n_col  = next((c for c in df_raw.columns if c[0] == src and 'liczba' in c[1]), None)
    med_col= next((c for c in df_raw.columns if c[0] == src and 'mediana' in c[1]), None)
    avg_col= next((c for c in df_raw.columns if c[0] == src and 'sredni' in c[1] and 'odch' not in c[1]), None)
    if n_col:
        records[f'n_{short}']      = pd.to_numeric(df_raw[n_col],   errors='coerce')
        records[f'median_{short}'] = pd.to_numeric(df_raw[med_col], errors='coerce') if med_col else np.nan
        records[f'mean_{short}']   = pd.to_numeric(df_raw[avg_col], errors='coerce') if avg_col else np.nan

df = pd.DataFrame(records)

# Normalize school name column (may have accent in some years)
if 'school_name' not in df.columns or df['school_name'].isna().all():
    name_col = next((c for c in df_raw.columns if c[0] == 'meta' and 'nazwa' in c[1]), None)
    if name_col:
        df['school_name'] = df_raw[name_col].values

df = df[df['n_polski'].notna() & (df['n_polski'] > 0)].copy()
print(f'Working dataset: {len(df):,} rows, {df["rspo"].nunique():,} unique schools')

## 2. Why only Polish, Maths, and English?

All students take Polish, Maths, and English. The other foreign languages
(French, German, Spanish, Russian, Italian) are taken by a small minority —
so we can't meaningfully compare schools on those.

The chart below shows the distribution of how many students took each subject per school per year.
A school with fewer than ~10 students in a given year gives statistically unreliable results
(one unusually talented or weak class can swing the school's score by 30+ percentile points).

In [ ]:
# ECDF of student counts per school-year for all subjects
all_subjects_short = {
    'polski':    'Polish',
    'matematyka':'Maths',
    'angielski': 'English',
    'francuski': 'French',
    'hiszpanski':'Spanish',
    'niemiecki': 'German',
    'rosyjski':  'Russian',
    'wloski':    'Italian',
}

core_short    = ['polski', 'matematyka', 'angielski']
minority_short= ['francuski', 'hiszpanski', 'niemiecki', 'rosyjski', 'wloski']

# ── ECDF for the 3 core subjects ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))

core_colors = ['#2166ac', '#d6604d', '#1a9850']
for short, color in zip(core_short, core_colors):
    col = f'n_{short}'
    if col not in df.columns:
        continue
    x = np.sort(df[col].dropna().values)
    y = np.arange(1, len(x) + 1) / len(x) * 100
    ax.plot(x, y, label=all_subjects_short[short], color=color, lw=2.5)

# Reference lines
for thresh, label in [(10, '10 students\n(25% of schools)'),
                       (20, '20 students\n(51% of schools)'),
                       (50, '50 students\n(78% of schools)')]:
    ax.axvline(thresh, color='#aaa', ls='--', lw=1)
    ax.text(thresh * 1.04, 103, label, fontsize=7.5, color='#666', va='bottom')

for ref in [25, 50, 75]:
    ax.axhline(ref, color='#ddd', lw=0.8)
    ax.text(302, ref, f'{ref}%', fontsize=8, color='#aaa', va='center', ha='left')

ax.set_xscale('log')
ticks = [1, 2, 3, 5, 10, 15, 20, 30, 50, 75, 100, 150, 200, 300]
ax.set_xticks(ticks)
ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
ax.set_xlim(1, 320)
ax.set_ylim(0, 110)
ax.set_yticks([0, 25, 50, 75, 100])
ax.yaxis.set_major_formatter(ticker.PercentFormatter(decimals=0))
ax.set_xlabel('Number of students per school per year (log scale)', fontsize=10)
ax.set_ylabel('% of schools', fontsize=10)
ax.set_title('Distribution of student counts — why only 3 subjects are usable', fontsize=11, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# ── Summary table for minority languages ─────────────────────────────────────
rows = []
for short in minority_short:
    col = f'n_{short}'
    if col not in df.columns:
        continue
    s = df[col].dropna()
    s = s[s > 0]
    rows.append({
        'Subject':           all_subjects_short[short],
        'School-year records': len(s),
        'Median students':   int(s.median()),
        'Max students':      int(s.max()),
    })
print('\nMinority languages — too few data points for reliable school comparison:')
display(pd.DataFrame(rows).set_index('Subject'))

**Takeaway:** Half of all schools have fewer than 20 students sitting any given exam in any given year.
For a quarter of schools, this number is below 10. This matters a lot for how we measure quality.

## 3. Choosing the best per-year metric

### The problem with raw scores

The exam difficulty changes year to year. In 2021, the national median for Maths was **46%**;
in 2022 it jumped to **60%**. A school scoring 55% in both years looks stable,
but in 2021 it was 9 points *above* average and in 2022 it was 5 points *below* average.
Raw scores alone can't tell us how a school is doing relative to its peers.

### Candidate metrics

For each school in each year we can compute:

| Metric | Description |
|--------|-------------|
| `mean` | Average student score |
| `median` | Middle student's score (less sensitive to outliers) |
| `pct_mean` | Percentile rank of the mean among all schools that year |
| `pct_median` | Percentile rank of the median among all schools that year |
| `diff_median` | School median minus national median for that year |
| `pct_above_national` | Is the school above the national median? (1 or 0) |

### How we choose

We use **jackknife (leave-one-out) stability**: for a school with 5 years of data,
we compute the metric 5 times, each time leaving out one year. The more the result
changes, the less stable the metric — and a less stable metric means the score is
driven by luck (which year happened to be included) rather than real school quality.

We want the metric where the 5 estimates are *closest together*.

In [ ]:
# Compute national median per year per subject (used for diff_median)
nat_median = {}
for short in core_short:
    nat_median[short] = df.groupby('year')[f'median_{short}'].median()
    df[f'nat_median_{short}'] = df['year'].map(nat_median[short])
    df[f'diff_median_{short}'] = df[f'median_{short}'] - df[f'nat_median_{short}']

# Compute percentile ranks within each year for mean and median
for short in core_short:
    for metric in ['median', 'mean']:
        col_in  = f'{metric}_{short}'
        col_out = f'pct_{metric}_{short}'
        df[col_out] = df.groupby('year')[col_in].transform(
            lambda x: stats.rankdata(x.fillna(x.median()), method='average') / len(x) * 100
        )

print('National median per year:')
nat_df = pd.DataFrame({SUBJECT_LABELS[f'jezyk {s}' if s != 'matematyka' else s]: 
                       nat_median[s] for s in ['polski', 'matematyka', 'angielski']})
nat_df.index.name = 'year'
display(nat_df.round(1))
print('\nNote: Maths swings 14 pp between 2021 and 2022 — a school with a constant raw score')
print('would look good in 2021 and mediocre in 2022 for no reason related to the school.')

In [ ]:
# ── Jackknife LOO stability ───────────────────────────────────────────────────
# Only use schools with data in at least 3 years
year_count = df.groupby('rspo')['year'].nunique()
df_multi = df[df['rspo'].isin(year_count[year_count >= 3].index)].copy()

METRIC_COLS = {
    'mean':             lambda s: f'mean_{s}',
    'median':           lambda s: f'median_{s}',
    'pct_mean':         lambda s: f'pct_mean_{s}',
    'pct_median':       lambda s: f'pct_median_{s}',
    'diff_median':      lambda s: f'diff_median_{s}',
    'pct_above_nat':    None,  # computed inline
}

loo_records = []

for rspo, grp in df_multi.groupby('rspo'):
    grp = grp.sort_values('year').reset_index(drop=True)
    k = len(grp)

    for short in core_short:
        n_vals = grp[f'n_{short}'].values
        if np.any(np.isnan(n_vals)) or n_vals.sum() == 0:
            continue

        loo_results = {m: [] for m in METRIC_COLS}

        for lo in range(k):
            mask  = np.ones(k, dtype=bool); mask[lo] = False
            n_sub = n_vals[mask]
            tot_n = n_sub.sum()
            if tot_n == 0:
                continue

            for metric_name, col_fn in METRIC_COLS.items():
                if metric_name == 'pct_above_nat':
                    vals = (grp[f'diff_median_{short}'].values[mask] > 0).astype(float)
                else:
                    col = col_fn(short)
                    vals = grp[col].values[mask]
                if np.any(np.isnan(vals)):
                    continue
                loo_results[metric_name].append(
                    (n_sub * vals).sum() / tot_n  # weighted mean across remaining years
                )

        row = {'rspo': rspo, 'subject': short, 'mean_n': n_vals.mean(), 'n_years': k}
        for metric_name, values in loo_results.items():
            if len(values) >= 2:
                row[f'loo_std_{metric_name}'] = np.std(values)
        loo_records.append(row)

loo = pd.DataFrame(loo_records)

# Size bins
loo['size_bin'] = pd.cut(
    loo['mean_n'],
    bins=[0, 9, 19, 49, 99, 9999],
    labels=['1–9', '10–19', '20–49', '50–99', '100+']
)

print(f'LOO computed for {len(loo):,} school-subject pairs')

In [ ]:
# ── Plot: ECDF of LOO std for each metric, for Maths (most variable) ──────────
loo_std_cols = [c for c in loo.columns if c.startswith('loo_std_')]
metric_display = {
    'loo_std_mean':          ('Mean score',              '#e41a1c', '-'),
    'loo_std_median':        ('Median score',            '#377eb8', '-'),
    'loo_std_pct_mean':      ('Percentile of mean',      '#ff7f00', '--'),
    'loo_std_pct_median':    ('Percentile of median',    '#984ea3', '--'),
    'loo_std_diff_median':   ('Diff from national median','#4daf4a', '-'),
    'loo_std_pct_above_nat': ('% years above national',  '#a65628', ':'),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, short in zip(axes, core_short):
    sub = loo[loo['subject'] == short]
    for col, (label, color, ls) in metric_display.items():
        if col not in sub.columns:
            continue
        x = np.sort(sub[col].dropna().values)
        y = np.arange(1, len(x) + 1) / len(x) * 100
        ax.plot(x, y, label=label, color=color, lw=2, ls=ls, alpha=0.9)

    ax.set_title(SUBJECT_LABELS[f'jezyk {short}' if short != 'matematyka' else 'matematyka'],
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('LOO std (lower = more stable)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_ylabel('% of schools', fontsize=9)
axes[0].yaxis.set_major_formatter(ticker.PercentFormatter(decimals=0))
axes[0].set_ylim(0, 105)

handles = [mlines.Line2D([], [], color=c, lw=2, ls=ls, label=lbl)
           for lbl, c, ls in metric_display.values()]
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=8.5,
           framealpha=0.9, bbox_to_anchor=(0.5, -0.04))

fig.suptitle(
    'Metric stability: ECDF of leave-one-out standard deviation\n'
    'A curve shifted LEFT means the metric is more stable (less sensitive to which year is included)',
    fontsize=10, fontweight='bold'
)
plt.tight_layout(rect=[0, 0.10, 1, 1])
plt.show()

In [ ]:
# ── Same, split by school size ────────────────────────────────────────────────
# Show Maths only (most variable; result pattern is the same for other subjects)
mat_loo = loo[loo['subject'] == 'matematyka'].copy()

SIZE_BINS = ['1–9', '10–19', '20–49', '50–99', '100+']
TOP_METRICS = ['loo_std_median', 'loo_std_diff_median', 'loo_std_pct_median']
TOP_COLORS  = ['#377eb8', '#4daf4a', '#984ea3']
TOP_LABELS  = ['Median score', 'Diff from national', 'Percentile of median']

fig, axes = plt.subplots(1, len(SIZE_BINS), figsize=(15, 4), sharey=False)

for ax, size_bin in zip(axes, SIZE_BINS):
    sub = mat_loo[mat_loo['size_bin'] == size_bin]
    n_schools = len(sub)
    for col, color, label in zip(TOP_METRICS, TOP_COLORS, TOP_LABELS):
        if col not in sub.columns:
            continue
        x = np.sort(sub[col].dropna().values)
        y = np.arange(1, len(x) + 1) / len(x) * 100
        ax.plot(x, y, color=color, lw=2, label=label)

    ax.set_title(f'{size_bin} students\n(n={n_schools})', fontsize=9, fontweight='bold')
    ax.set_xlabel('LOO std', fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

axes[0].set_ylabel('% of schools', fontsize=9)
axes[0].yaxis.set_major_formatter(ticker.PercentFormatter(decimals=0))

handles = [mlines.Line2D([], [], color=c, lw=2, label=lbl)
           for c, lbl in zip(TOP_COLORS, TOP_LABELS)]
fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=9,
           framealpha=0.9, bbox_to_anchor=(0.5, -0.04))

fig.suptitle(
    'Maths — metric stability by school size\n'
    'The winner changes depending on how many students the school has',
    fontsize=10, fontweight='bold'
)
plt.tight_layout(rect=[0, 0.12, 1, 1])
plt.show()

In [ ]:
# ── Summary table: median LOO std per size bin ────────────────────────────────
agg = (
    mat_loo.groupby('size_bin', observed=True)[TOP_METRICS]
    .median()
    .rename(columns=dict(zip(TOP_METRICS, TOP_LABELS)))
    .round(2)
)
print('Median LOO std for Maths — lower is better:')
display(agg.style.highlight_min(axis=1, color='#c6efce')
             .set_caption('Green = most stable metric for that school size'))

**Finding:** `diff_median` (school median minus the national median for that year)
is the most stable metric for schools with **≥ 10 students**,
which covers about 75% of schools.

For very small schools (< 10 students) the raw median score is marginally more stable,
but the difference is tiny (< 0.15 pp). We will use `diff_median` for all schools
because it removes the year-difficulty bias — a school that happened to miss
the difficult 2021 Maths exam (national median 46 pp vs. 60 pp in 2022)
would get a **2.9 pp bonus** with raw scores but only **0.16 pp** with `diff_median`.

## 4. Aggregating across years

We have up to 5 years of data per school. We need to combine them into one number.

Three natural options:
- **Weighted mean** — weight each year by the number of students sitting the exam
- **Median of years** — take the middle year's result (ignores how many students)
- **Trimmed mean** — drop the best and worst year, average the rest

We apply the same jackknife test: which aggregation gives the most stable
school score when we drop one year?

In [ ]:
# For schools with 5 years of data: LOO stability of 3 aggregation methods
year_count_5 = df.groupby('rspo')['year'].nunique()
df5 = df[df['rspo'].isin(year_count_5[year_count_5 == 5].index)].copy()

agg_records = []

for rspo, grp in df5.groupby('rspo'):
    grp = grp.sort_values('year').reset_index(drop=True)

    for short in core_short:
        n_vals   = grp[f'n_{short}'].values
        diff_vals= grp[f'diff_median_{short}'].values
        if np.any(np.isnan(n_vals)) or np.any(np.isnan(diff_vals)):
            continue

        loo_wmean, loo_median, loo_trimmed = [], [], []

        for lo in range(5):
            mask  = np.ones(5, dtype=bool); mask[lo] = False
            n_sub = n_vals[mask]
            d_sub = diff_vals[mask]
            tot_n = n_sub.sum()
            if tot_n == 0:
                continue
            loo_wmean.append((n_sub * d_sub).sum() / tot_n)
            loo_median.append(np.median(d_sub))
            # trimmed: with 4 values, drop min and max (keep middle 2)
            sorted_d = np.sort(d_sub)
            loo_trimmed.append(sorted_d[1:3].mean())

        if len(loo_wmean) >= 4:
            agg_records.append({
                'rspo':            rspo,
                'subject':         short,
                'mean_n':          n_vals.mean(),
                'std_wmean':       np.std(loo_wmean),
                'std_median_yrs':  np.std(loo_median),
                'std_trimmed':     np.std(loo_trimmed),
            })

agg_loo = pd.DataFrame(agg_records)
agg_loo['size_bin'] = pd.cut(
    agg_loo['mean_n'],
    bins=[0, 9, 19, 49, 99, 9999],
    labels=['1–9', '10–19', '20–49', '50–99', '100+']
)

agg_summary = (
    agg_loo.groupby('size_bin', observed=True)[['std_wmean', 'std_median_yrs', 'std_trimmed']]
    .median()
    .rename(columns={
        'std_wmean':      'Weighted mean',
        'std_median_yrs': 'Median of years',
        'std_trimmed':    'Trimmed mean',
    })
    .round(3)
)
print('Median LOO std for aggregation methods (diff_median as input), all subjects:')
display(agg_summary.style.highlight_min(axis=1, color='#c6efce')
                   .set_caption('Green = most stable aggregation for that school size'))

**Finding:** Weighted mean (weighting each year by number of students)
is the most stable or tied-most-stable for all school sizes.
It also has the best intuitive justification: a year where 50 students sat the exam
should count more than a year where only 5 did.

**Final per-subject score formula:**
```
score(school, subject) = Σ(n_year × diff_year) / Σ(n_year)

where diff_year = school_median_year − national_median_year
```
This gives a number in percentage points: positive means above average for the country,
negative means below average, zero means exactly at the national median.

In [ ]:
# ── Compute final per-subject score for every school ──────────────────────────
score_records = []

for rspo, grp in df.groupby('rspo'):
    row = {'rspo': rspo}
    row['school_name'] = grp['school_name'].iloc[0]
    row['is_public']   = grp['is_public'].iloc[0]
    row['n_years']     = grp['year'].nunique()
    row['years']       = sorted(grp['year'].unique().tolist())

    for short in core_short:
        n_vals   = pd.to_numeric(grp[f'n_{short}'], errors='coerce')
        diff_vals= pd.to_numeric(grp[f'diff_median_{short}'], errors='coerce')
        valid    = n_vals.notna() & diff_vals.notna() & (n_vals > 0)
        if valid.sum() == 0:
            continue
        n_v = n_vals[valid].values
        d_v = diff_vals[valid].values
        tot_n = n_v.sum()
        row[f'score_{short}']   = (n_v * d_v).sum() / tot_n
        row[f'n_total_{short}'] = int(tot_n)
        row[f'n_years_{short}'] = int(valid.sum())

    score_records.append(row)

scores = pd.DataFrame(score_records)
scores = scores.dropna(subset=[f'score_{s}' for s in core_short])

# Add percentile ranks (used for the map colour)
for short in core_short:
    scores[f'pct_{short}'] = (
        stats.rankdata(scores[f'score_{short}'], method='average') / len(scores) * 100
    )

# Sigma per subject (for colour scale)
sigma = {short: scores[f'score_{short}'].std() for short in core_short}
print('Sigma per subject (used for colour saturation thresholds):')
for short in core_short:
    subj_name = SUBJECT_LABELS.get(f'jezyk {short}' if short != 'matematyka' else 'matematyka',
                                    short)
    print(f'  {subj_name:10s}: σ = {sigma[short]:.1f} pp   \'saturate\' at ±{1.5*sigma[short]:.0f} pp')

print(f'\nSchools with scores: {len(scores):,}')

## 5. Which schools are consistently at the top?

Because individual years are noisy, we want to check that the top schools
are not just lucky — they should appear at the top across *multiple* LOO folds.

For each school we compute: how often does it appear in the top N
when we drop one year at a time? A school that is *always* in the top 30
is more trustworthy than one that is there only when one particular year is included.

In [ ]:
# Jackknife stability of school ranking (Maths, schools with 5 years)
TOP_N_LIST = [5, 10, 30, 50]

def loo_scores(rspo_grp_iter, short: str) -> dict:
    """Return {rspo: [score_loo_0, score_loo_1, ...]} for all schools."""
    result = {}
    for rspo, grp in rspo_grp_iter:
        grp = grp.sort_values('year').reset_index(drop=True)
        n_vals   = grp[f'n_{short}'].values
        diff_vals= grp[f'diff_median_{short}'].values
        if np.any(np.isnan(n_vals)) or np.any(np.isnan(diff_vals)):
            continue
        k = len(grp)
        loos = []
        for lo in range(k):
            mask  = np.ones(k, dtype=bool); mask[lo] = False
            n_sub = n_vals[mask]; tot_n = n_sub.sum()
            if tot_n == 0: continue
            loos.append((n_sub * diff_vals[mask]).sum() / tot_n)
        if loos:
            result[rspo] = loos
    return result


short = 'matematyka'  # show for Maths; pattern is similar for other subjects
loo_vals = loo_scores(df5.groupby('rspo'), short)

# For each LOO fold, rank all schools and record top-N membership
n_folds = 5
all_rspo = list(loo_vals.keys())

# Matrix: rows=schools, cols=folds, value=score
fold_matrix = pd.DataFrame(
    {rspo: {f: vals[f] if f < len(vals) else np.nan for f in range(n_folds)}
     for rspo, vals in loo_vals.items()}
).T  # shape: (n_schools, 5)

consistency_rows = []
name_lu = df5.groupby('rspo')['school_name'].first()

for rspo in fold_matrix.index:
    row = {'rspo': rspo, 'school_name': name_lu.get(rspo, '?')}
    full_score = fold_matrix.loc[rspo].mean()  # mean across LOO folds ≈ full score
    row['mean_score'] = full_score
    for top_n in TOP_N_LIST:
        count_in_top = 0
        for fold in range(n_folds):
            fold_scores = fold_matrix[fold].dropna()
            rank = (fold_scores > fold_matrix.loc[rspo, fold]).sum() + 1
            if rank <= top_n:
                count_in_top += 1
        row[f'in_top_{top_n}_of_{n_folds}'] = count_in_top
    consistency_rows.append(row)

consistency = pd.DataFrame(consistency_rows).sort_values('mean_score', ascending=False)

print(f'Schools consistently in the top (Maths, schools with 5 years of data):')
display(
    consistency.head(40)[[
        'school_name', 'mean_score',
        f'in_top_5_of_{n_folds}', f'in_top_10_of_{n_folds}',
        f'in_top_30_of_{n_folds}', f'in_top_50_of_{n_folds}',
    ]].rename(columns={
        'mean_score':                'Avg score (pp above national)',
        f'in_top_5_of_{n_folds}':   f'In top 5 / {n_folds} folds',
        f'in_top_10_of_{n_folds}':  f'In top 10 / {n_folds} folds',
        f'in_top_30_of_{n_folds}':  f'In top 30 / {n_folds} folds',
        f'in_top_50_of_{n_folds}':  f'In top 50 / {n_folds} folds',
    }).style.background_gradient(cmap='Greens', subset=[f'In top 5 / {n_folds} folds',
                                                         f'In top 10 / {n_folds} folds'])
)

## 6. Combining all three subjects

A school could score well in Maths but poorly in Polish. How do we combine
the three subject scores into a single number for the map?

First, we check how correlated the three subjects are.
If they are highly correlated, the choice of combination method barely matters.
If they are independent, we need to be careful.

In [ ]:
# ── Subject correlation ───────────────────────────────────────────────────────
score_cols = [f'score_{s}' for s in core_short]
corr = scores[score_cols].rename(
    columns={f'score_{s}': SUBJECT_LABELS.get(f'jezyk {s}' if s != 'matematyka' else 'matematyka', s)
             for s in core_short}
).corr()

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=0, vmax=1, linewidths=0.5, ax=ax,
    annot_kws={'size': 13, 'weight': 'bold'}
)
ax.set_title('Correlation between subject scores\n(across all schools)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nSpearman rank correlation (more robust to outliers):')
print(scores[score_cols].rename(
    columns={f'score_{s}': SUBJECT_LABELS.get(f'jezyk {s}' if s != 'matematyka' else 'matematyka', s)
             for s in core_short}
).corr(method='spearman').round(2))

In [ ]:
# ── Composite score: mean of per-subject percentiles ─────────────────────────
# We use percentile ranks (not raw pp) so that differences in subject difficulty
# don't inflate one subject's weight.
# (e.g. English has σ≈19pp, Polish has σ≈9pp — a school 10pp above average in English
#  and 10pp above average in Polish is at a higher percentile in Polish)

scores['composite_pct'] = scores[[f'pct_{s}' for s in core_short]].mean(axis=1)

# Flag: good in all 3 subjects (above the 60th percentile in each)
scores['good_in_all_3'] = (
    (scores['pct_polski']    >= 60) &
    (scores['pct_matematyka']>= 60) &
    (scores['pct_angielski'] >= 60)
)

print(f'Schools good in all 3 subjects (≥60th percentile each): {scores["good_in_all_3"].sum()}')
print(f'  of which public:   {(scores["good_in_all_3"] & (scores["is_public"]=="Tak")).sum()}')
print(f'  of which private:  {(scores["good_in_all_3"] & (scores["is_public"]=="Nie")).sum()}')

# Distribution of composite score
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

subjects_to_plot = [(f'score_{s}', SUBJECT_LABELS.get(f'jezyk {s}' if s != 'matematyka' else 'matematyka', s),
                     SUBJECT_COLORS[f'jezyk {s}' if s != 'matematyka' else 'matematyka'])
                    for s in core_short]
subjects_to_plot.append(('composite_pct', 'Composite (mean pct)', '#555555'))

for ax, (col, label, color) in zip(axes, subjects_to_plot):
    data = scores[col].dropna()
    ax.hist(data, bins=40, color=color, alpha=0.75, edgecolor='white', lw=0.3)
    ax.axvline(data.median(), color='black', lw=1.5, ls='--', label=f'median={data.median():.1f}')
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel('pp above/below national' if 'score' in col else 'percentile', fontsize=8)
    ax.legend(fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle('Distribution of school scores', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Final metric definition

Based on the analysis above, here is the metric used for the school quality map.

### Per-subject score

```
diff_year(school, subject, year) =
    median_score(school, subject, year) − national_median(subject, year)

score(school, subject) =
    Σ_year [ n_students(year) × diff_year ] / Σ_year [ n_students(year) ]
```

This gives a number in percentage points:
- **+10 pp** means the middle student at this school scores 10 points above
  the middle student nationally, averaged over all available years.
- **−5 pp** means the school is below average by 5 points.

### Why this metric?

1. **Difference from national median** removes the year-difficulty effect
   (the 14 pp swing in Maths difficulty between 2021 and 2022 is neutralised).
2. **Median** is preferred over mean because outlier students
   (one gifted or one struggling child) affect the mean more than the median.
3. **Weighted by student count** means a year with 50 students carries
   more weight than a year with 5, which is statistically correct.
4. The metric is **not a percentile** — a school 10 pp above average gets that
   number regardless of whether 300 or 3 other schools are also around that level.
   Percentile is used only for colouring the map markers.

### Map colour scale

The marker colour on the map is a diverging green–yellow–red gradient:
- **Yellow** = within ±0.33σ of the national median (not meaningfully different)
- **Deep green** = ≥ +1.5σ above national (reliably excellent)
- **Deep red** = ≤ −1.5σ below national (reliably weak)

Schools far above +1.5σ all get the same deep green — they are all excellent,
and the difference between them is within statistical noise given the sample sizes.

### Composite score

The map also shows a composite score: the mean of the three subject percentile ranks.
Schools flagged as **good in all 3** (≥60th percentile in each subject)
are highlighted separately — this is the most conservative indicator of
all-round quality.

In [ ]:
# ── Export for the map ────────────────────────────────────────────────────────
export_cols = [
    'rspo', 'school_name', 'is_public', 'n_years',
    'score_polski',    'pct_polski',    'n_total_polski',
    'score_matematyka','pct_matematyka','n_total_matematyka',
    'score_angielski', 'pct_angielski', 'n_total_angielski',
    'composite_pct',   'good_in_all_3',
]
export = scores[[c for c in export_cols if c in scores.columns]].copy()

# Sigma values needed by the map for colour scale
sigma_note = pd.Series({
    'sigma_polski':     round(sigma['polski'],    2),
    'sigma_matematyka': round(sigma['matematyka'],2),
    'sigma_angielski':  round(sigma['angielski'], 2),
})
print('Sigma values for map colour scale:')
print(sigma_note.to_string())
print()

out_path = Path('../data/school_scores.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
export.to_csv(out_path, index=False)
print(f'Exported {len(export):,} schools to {out_path}')
display(export.head(3))